In [6]:
from langchain_deepseek import ChatDeepSeek
import os

os.environ["DEEPSEEK_API_KEY"] = "sk-49892f84100b4c73843a82e371fc3522"

model = ChatDeepSeek(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)
model

ChatDeepSeek(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3', 'langchain-openai': '1.6.7', 'langchain-deepseek': '1.1.1'}}, profile={'name': 'DeepSeek V4.1 Flash', 'release_date': '2026-09-10', 'last_updated': '2026-09-10', 'open_weights': True, 'max_input_tokens': 1000000, 'max_output_tokens': 384000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x128ae2ed0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x128f7e0c0>, root_client=<openai.OpenAI object at 0x128ae24b0>, root_async_client=<openai.AsyncOpenAI object at 0x128f7c3e0>, model_name='deepseek-flash', model_kw

In [7]:
import json

json_schema = {
    "title": "Movie",
    "description": "A movie with details",
    "type": "object",
    "properties": {
        "title": {
            "type": "string",
            "description": "The title of the movie"
        },
        "year": {
            "type": "integer",
            "description": "The year the movie was released"
        },
        "director": {
            "type": "string",
            "description": "The director of the movie"
        },
        "rating": {
            "type": "number",
            "description": "The movie's rating out of 10"
        }
    },
    "required": ["title", "year", "director", "rating"]
}

model_with_structure = model.with_structured_output(
    json_schema,
    method="json_schema",
)
response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)  # {'title': 'Inception', 'year': 2010, ...}

{'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.8}


In [8]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("Provide details about the movie Inception")
response
# {
#     "raw": AIMessage(...),
#     "parsed": Movie(title=..., year=..., ...),
#     "parsing_error": None,
# }

{'raw': AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 85, 'prompt_tokens': 355, 'total_tokens': 440, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 355}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': 'a3e72de0-fafa-4e34-88df-0e10c2d35751', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0f70e-eb8e-7631-8449-c7a15eecb8fa-0', tool_calls=[{'name': 'Movie', 'args': {'title': 'Inception', 'year': 2010, 'director': 'Christopher Nolan', 'rating': 8.8}, 'id': 'call_00_bblB0sIzRXOXSizTXL1l6983', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 355, 'output_tokens': 85, 'total_tokens': 440, 'input_token_details': {'cache_

In [9]:
response = model.invoke("Create a picture of a cat")
print(response.content_blocks)

[{'type': 'text', 'text': 'I\'m not able to generate or create images directly. However, I can describe a cat for you in detail, or help you create one using other tools!\n\nHere are some options:\n\n1. **AI Image Generators** - You can use tools like:\n   - DALL·E (via ChatGPT Plus)\n   - Midjourney\n   - Stable Diffusion\n   - Adobe Firefly\n   - Bing Image Creator (free)\n\n2. **Text-based art** - Here\'s a simple ASCII cat:\n```\n /\\_/\\\n( o.o )\n > ^ <\n```\n\n3. **A detailed description** you could use as a prompt for an image generator:\n> "A fluffy orange tabby cat sitting on a sunlit windowsill, green eyes gazing curiously outside, soft fur glowing in the warm afternoon light, cozy home interior blurred in the background, photorealistic style."\n\nWould you like me to help you craft a prompt for an image generator, or write a poem/story about a cat instead?'}]
